#### Importation des modules

In [77]:
import os
import sys
import requests
import datetime 
import random
from pathlib import Path
import json
import time

import pandas as pd
import ta
from faker import Faker
import uuid
import sqlite3
from decimal import Decimal

sys.path.append(str(Path().resolve().parent / "utils"))

from utils import get_data, convert_in_ms, get_crypto_price, get_historical_data
from client import Client
from simulate_order import simulate_order

### Backtest avec MMS combiner avec un modèle de machine learning

In [70]:
df = pd.read_json(r"../etape_1_exploration_api/BNBUSDT_1h.json")
df = df.set_index("open_time")

In [71]:
df["SMA200"] = ta.trend.sma_indicator(df["close"], 200)

df["SMA600"] = ta.trend.sma_indicator(df["close"], 600)

In [72]:
df.tail(5)

,open,high,low,close,volume,SMA200,SMA600
open_time,,,,,,,
2026-07-04 20:00:00,575.52,575.95,573.16,573.21,3048.389,558.69075,580.841000
2026-07-04 21:00:00,573.21,575.47,572.57,574.68,2820.620,558.74475,580.807867
2026-07-04 22:00:00,574.68,578.27,574.49,576.67,3193.494,558.82910,580.780833
2026-07-04 23:00:00,576.68,576.80,573.70,575.37,3195.293,558.87955,580.750717
2026-07-05 00:00:00,575.38,575.43,572.15,573.26,2964.556,558.91170,580.717100


In [73]:
usdt = 1000
btc = 0

latest_index = df.first_valid_index()

In [74]:
for index, row in df.iterrows():
    if df["SMA200"][latest_index] >  df["SMA600"][latest_index] and usdt > 10:
        btc = usdt /  df["close"][index]
        btc = btc - 0.007 * btc
        usdt = 0
        print("Buy btc at", df["close"][index], "$ the", index)
    if df["SMA200"][latest_index] <  df["SMA600"][latest_index] and btc > 0.0001:
        usdt = btc *  df["close"][index]
        usdt = usdt - 0.007 * usdt
        btc = 0
        print("Sell btc at", df["close"][index], "$ the", index)
    latest_index = index

Buy btc at 1.98 $ the 2017-12-01 03:00:00
Sell btc at 12.8394 $ the 2018-01-23 15:00:00
Buy btc at 10.729 $ the 2018-02-20 14:00:00
Sell btc at 8.4 $ the 2018-03-08 16:00:00
Buy btc at 12.1323 $ the 2018-03-24 07:00:00
Sell btc at 12.7042 $ the 2018-05-12 21:00:00
Buy btc at 12.7945 $ the 2018-05-25 06:00:00
Sell btc at 12.6572 $ the 2018-05-27 14:00:00
Buy btc at 14.3778 $ the 2018-06-04 03:00:00
Sell btc at 13.99 $ the 2018-06-27 20:00:00
Buy btc at 14.0769 $ the 2018-07-29 19:00:00
Sell btc at 11.61 $ the 2018-08-11 14:00:00
Buy btc at 11.4144 $ the 2018-09-01 17:00:00
Sell btc at 9.463 $ the 2018-09-10 22:00:00
Buy btc at 10.0255 $ the 2018-09-27 22:00:00
Sell btc at 9.6267 $ the 2018-10-15 00:00:00
Buy btc at 5.6522 $ the 2018-12-22 14:00:00
Sell btc at 19.0699 $ the 2019-05-09 20:00:00
Buy btc at 25.2194 $ the 2019-05-17 22:00:00
Sell btc at 31.5169 $ the 2019-06-10 12:00:00
Buy btc at 32.8298 $ the 2019-06-15 21:00:00
Sell btc at 32.488 $ the 2019-07-03 09:00:00
Buy btc at 30.13

### Comparaison avec la stratégie buy and hold

In [75]:
final_result = usdt + btc * df["close"].iloc[-1]
print("Final result", final_result, "USDT")

Final result 131126.22090113477 USDT


In [76]:
print('Result of buy and hold is', (1000 / df["close"].iloc[0]) * df["close"].iloc[-1], 'USDT')

Result of buy and hold is 337211.76470588235 USDT


### Trade dans un cas réel

In [17]:
DIR = Path().resolve().parent / "data"
os.chdir(DIR)

In [ ]:
MARGE_SECURITE_FRAIS = Decimal("0.999")
 
 
def update_balance(client, order):
    executed_qty = Decimal(str(order["executedQty"]))
    quote_qty = Decimal(str(order["cummulativeQuoteQty"]))
 
    frais_crypto = sum(
        Decimal(str(f["commission"])) for f in order["fills"]
        if f["commissionAsset"] == client.crypto
    )
    frais_usdt = sum(
        Decimal(str(f["commission"])) for f in order["fills"]
        if f["commissionAsset"] == "USDT"
    )
 
    if order["side"] == "BUY":
        client.usdt -= float(quote_qty)
        client.usdt -= float(frais_usdt)
        client.solde_crypto += float(executed_qty)
        client.solde_crypto -= float(frais_crypto)
    elif order["side"] == "SELL":
        client.solde_crypto -= float(executed_qty)
        client.solde_crypto -= float(frais_crypto)
        client.usdt += float(quote_qty)
        client.usdt -= float(frais_usdt)
 
    client.save()
    return client
 
 
def save_order(order, path="transactions.jsonl"):
    with open(path, "a") as f:
        f.write(json.dumps(order) + "\n")
 
 
def trade(symbol, client):
    data = get_historical_data(symbol, "1h", 650)
    df = pd.DataFrame(data)
 
    df["SM200"] = ta.trend.sma_indicator(df["close"], 200)
    df["SM600"] = ta.trend.sma_indicator(df["close"], 600)
 
    signal_achat = df["SM200"].iloc[-2] > df["SM600"].iloc[-2]
    signal_vente = df["SM200"].iloc[-2] < df["SM600"].iloc[-2]
 
    order = None
 
    if client.usdt > 5 and signal_achat:
        quantity = (client.usdt * float(MARGE_SECURITE_FRAIS)) / df["close"].iloc[-1]
        order = client.buy_or_sell("BUY", symbol, df, quantity)
        update_balance(client, order)
 
    elif client.solde_crypto > 0.0001 and signal_vente:
        quantity = client.solde_crypto
        order = client.buy_or_sell("SELL", symbol, df, quantity)
        update_balance(client, order)
 
    if order:
        save_order(order)
 
    return client
 

In [ ]:
client = Client()  # demande nom, montant, crypto une seule fois
symbol = f"{client.crypto}USDT"
while True:
    client = trade(symbol, client)
    time.sleep(3600)

Solde existant chargé pour egah epiphane (700.00 USDT / 0.000000 BTC)
